# TartanGround Dataset Downloader & Reformatter for Google Colab
This notebook automates downloading the TartanGround dataset using the official `tartanairpy` package and saving it directly to your mounted Google Drive in the format expected by the `bridgedepth` data loader.

## Key Optimizations & Design:
1. **Targeted Front View Only**: Configured to download only the **front camera view** (`lcam_front`, `rcam_front`) and **front disparity/depth** (`depth_lcam_front`). This saves massive amount of space (only ~15% of the full dataset size) and is fully supported by the robust dataloader.
2. **Parallel Downloading**: Utilizes `tartanairpy`'s multi-threaded downloader (`download_ground_multi_thread`) to download zip archives fast.
3. **Zero Local Storage Overhead**: Downloads are processed sequentially environment-by-environment. After each environment is extracted and copied to Google Drive, the local VM files are deleted and `drive.flush_and_unmount()` is called to flush and purge the DriveFS FUSE write cache (`/root/.config/Google/DriveFS`), reclaiming 100% of the local SSD space before starting the next environment.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Install tartanairpy package directly from GitHub to get the latest downloader features
!pip install git+https://github.com/castacks/tartanairpy.git

In [ ]:
import os
import shutil
import time
import tartanair as ta

# ── Target Directory inside Google Drive ──────────────────────────────────
# The bridgedepth data loader expects: 
#  - datasets/TartanGround/
GDRIVE_TARTANGROUND_ROOT = "/content/drive/MyDrive/TartanGround"

# Sensor modalities needed by the dataloader
MODALITIES = ['image', 'depth']

# Robot platforms to download ('omni', 'diff', 'anymal'). 'omni' is standard.
VERSIONS = ['omni']

# Camera views required by bridgedepth (restricting to lcam_front and rcam_front only)
CAMERAS = [
    'lcam_front', 'rcam_front'
]

# Environments to process. You can edit this list to add more or disable environments.
# TartanGround contains 60+ environments. A subset is configured here by default.
ENVIRONMENTS = [
    {"name": "AbandonedCable", "enabled": True},
    {"name": "AbandonedFactory", "enabled": True},
    {"name": "AbandonedFactory2", "enabled": True},
    {"name": "AbandonedSchool", "enabled": True},
    {"name": "CarWelding", "enabled": True},
    {"name": "ModNeighborhood", "enabled": True},
    {"name": "OldtownSummer", "enabled": True},
    {"name": "NordicHarbor", "enabled": True},
    {"name": "ForestAutumn", "enabled": True},
    {"name": "ForestWinter", "enabled": True},
    {"name": "GreatMarsh", "enabled": True},
    {"name": "AmericanDiner", "enabled": True},
    {"name": "ArchVizTinyHouseDay", "enabled": True},
    {"name": "ArchVizTinyHouseNight", "enabled": True},
    {"name": "Antiquity3D", "enabled": True},
    {"name": "Apocalyptic", "enabled": True},
    {"name": "BrushifyMoon", "enabled": True},
]

In [ ]:
import os
import glob
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys

def validate_npy_file(npy_path):
    try:
        np.load(npy_path, mmap_mode='r')
        return True
    except Exception as e:
        print(f"\n[WARNING] Corrupted NPY file detected at {npy_path}: {e}. Removing it.")
        try:
            os.remove(npy_path)
        except:
            pass
        return False

def validate_trajectory(traj_path):
    left_imgs = sorted(glob.glob(os.path.join(traj_path, "image_lcam_front/*.png")))
    right_imgs = sorted(glob.glob(os.path.join(traj_path, "image_rcam_front/*.png")))
    depth_npys = sorted(glob.glob(os.path.join(traj_path, "depth_lcam_front/*.npy")))
    depth_pngs = sorted(glob.glob(os.path.join(traj_path, "depth_lcam_front/*.png")))
    
    # Filter and validate existing NPY files
    valid_npys = [npy for npy in depth_npys if validate_npy_file(npy)]
    
    num_left = len(left_imgs)
    num_right = len(right_imgs)
    num_npy = len(valid_npys)
    num_png = len(depth_pngs)
    
    if num_left == 0:
        return False, "No left images found"
    if num_right == 0:
        return False, "No right images found"
    if num_npy + num_png == 0:
        return False, "No depth maps found"
        
    if num_left != num_right:
        return False, f"Mismatch in image counts: left={num_left}, right={num_right}"
    if num_left != (num_npy + num_png):
        return False, f"Mismatch in depth counts: images={num_left}, npy={num_npy}, png={num_png}"
        
    # Check if basenames match exactly
    left_basenames = {os.path.splitext(os.path.basename(f))[0] for f in left_imgs}
    right_basenames = {os.path.splitext(os.path.basename(f))[0] for f in right_imgs}
    depth_basenames = {os.path.splitext(os.path.basename(f))[0] for f in valid_npys + depth_pngs}
    
    if left_basenames != right_basenames:
        return False, "Left and right image file names do not match"
    if left_basenames != depth_basenames:
        return False, "Image and depth file names do not match"
        
    return True, "Valid"

def validate_environment(env_path):
    traj_paths = sorted(glob.glob(os.path.join(env_path, "Data_omni/P*")))
    if not traj_paths:
        return False, "No trajectories found"
        
    for traj in traj_paths:
        is_valid, reason = validate_trajectory(traj)
        if not is_valid:
            return False, f"Trajectory {os.path.basename(traj)} invalid: {reason}"
            
    return True, "Valid"

def preprocess_single_depth(png_path):
    try:
        # Load and decode depth
        depth_rgba = cv2.imread(png_path, cv2.IMREAD_UNCHANGED)
        if depth_rgba is None:
            return png_path, False, "Could not read image"
        depth = depth_rgba.view("<f4")
        depth = np.squeeze(depth, axis=-1)
        
        # Save to .npy
        npy_path = png_path[:-4] + ".npy"
        np.save(npy_path, depth)
        
        # Delete original png
        os.remove(png_path)
        return png_path, True, None
    except Exception as e:
        return png_path, False, str(e)

def preprocess_depth_files(env_dir):
    depth_pngs = glob.glob(os.path.join(env_dir, "**/depth_lcam_front/**/*.png"), recursive=True)
    if not depth_pngs:
        return
    total = len(depth_pngs)
    print(f"[preprocessor] Preprocessing {total} depth PNGs to NPY in {env_dir} using 16 threads...")
    
    count = 0
    failed = 0
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(preprocess_single_depth, p): p for p in depth_pngs}
        for i, future in enumerate(as_completed(futures)):
            png_path, success, err = future.result()
            if success:
                count += 1
            else:
                failed += 1
                print(f"\n[WARNING] Failed to preprocess {png_path}: {err}")
            
            percent = (i + 1) * 100 // total
            sys.stdout.write(f"\r[preprocessor] Progress: {i+1}/{total} ({percent}%) | Success: {count} | Failed: {failed}")
            sys.stdout.flush()
    print(f"\n[preprocessor] Successfully preprocessed {count}/{total} depth files.")

def visualize_sample(env_path):
    img_left_files = sorted(glob.glob(os.path.join(env_path, "**/image_lcam_front/*.png"), recursive=True))
    if not img_left_files:
        print("[vis] No lcam_front images found for visualization.")
        return
    left_path = img_left_files[0]
    right_path = left_path.replace("lcam_front", "rcam_front")
    depth_path = left_path.replace("image_lcam_front", "depth_lcam_front").replace(".png", ".npy")
    
    if not os.path.exists(right_path):
        print(f"[vis] Right image not found at {right_path}")
        return
    if not os.path.exists(depth_path):
        depth_path_png = left_path.replace("image_lcam_front", "depth_lcam_front")
        if os.path.exists(depth_path_png):
            depth_path = depth_path_png
        else:
            print(f"[vis] Depth file not found at {depth_path} or {depth_path_png}")
            return
            
    print(f"[vis] Visualizing sample:")
    print(f"  Left image: {left_path}")
    print(f"  Right image: {right_path}")
    print(f"  Depth map: {depth_path}")
    
    left_img = cv2.cvtColor(cv2.imread(left_path), cv2.COLOR_BGR2RGB)
    right_img = cv2.cvtColor(cv2.imread(right_path), cv2.COLOR_BGR2RGB)
    
    if depth_path.endswith('.npy'):
        depth = np.load(depth_path)
    else:
        depth_rgba = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
        depth = depth_rgba.view("<f4")
        depth = np.squeeze(depth, axis=-1)
        
    disp = 80.0 / (depth + 1e-6)
    disp[depth <= 0] = 0
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(left_img)
    axes[0].set_title("Left Image (lcam_front)")
    axes[0].axis('off')
    
    axes[1].imshow(right_img)
    axes[1].set_title("Right Image (rcam_front)")
    axes[1].axis('off')
    
    im = axes[2].imshow(disp, cmap='plasma')
    axes[2].set_title("Disparity (from Depth NPY)")
    axes[2].axis('off')
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()

if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(GDRIVE_TARTANGROUND_ROOT, exist_ok=True)

for idx, env_info in enumerate(ENVIRONMENTS):
    env_name = env_info["name"]
    if not env_info["enabled"]:
        print(f"\n>>> Skipping environment {idx+1}/{len(ENVIRONMENTS)}: {env_name}")
        continue
        
    print("\n" + "=" * 80)
    print(f"  PROCESSING ENVIRONMENT {idx+1}/{len(ENVIRONMENTS)}: {env_name}")
    print("=" * 80)
    
    # 0. Check and mount drive if unmounted
    if not os.path.exists("/content/drive/MyDrive"):
        print("[gdrive] Mount not active. Remounting Google Drive...")
        drive.mount('/content/drive')
        os.makedirs(GDRIVE_TARTANGROUND_ROOT, exist_ok=True)
        
    # Final destination path for this environment
    dest_env_path = os.path.join(GDRIVE_TARTANGROUND_ROOT, env_name)
    
    # Run a rigorous validation check on images, NPYs, and file numbering
    is_valid, reason = validate_environment(dest_env_path)
    if is_valid:
        print(f"[skip] Environment '{env_name}' is fully completed, verified, and preprocessed. Skipping all steps.")
        visualize_sample(dest_env_path)
        continue
    else:
        print(f"[info] Environment '{env_name}' verification check: {reason}. Processing needed.")
        
    try:
        # Clean up any leftover right camera depth folders
        import shutil
        for rcam_depth_folder in glob.glob(os.path.join(dest_env_path, "**/depth_rcam_front"), recursive=True):
            print(f"[cleanup] Removing existing right camera depth folder: {rcam_depth_folder}")
            shutil.rmtree(rcam_depth_folder, ignore_errors=True)
            
        # 1. Check if download is already done
        traj_paths = sorted(glob.glob(os.path.join(dest_env_path, "Data_omni/P*")))
        download_needed = False
        if not traj_paths:
            download_needed = True
        else:
            for traj in traj_paths:
                left_imgs = glob.glob(os.path.join(traj, "image_lcam_front/*.png"))
                right_imgs = glob.glob(os.path.join(traj, "image_rcam_front/*.png"))
                depth_npys = [npy for npy in glob.glob(os.path.join(traj, "depth_lcam_front/*.npy")) if validate_npy_file(npy)]
                depth_pngs = glob.glob(os.path.join(traj, "depth_lcam_front/*.png"))
                
                num_left = len(left_imgs)
                num_right = len(right_imgs)
                num_depth = len(depth_npys) + len(depth_pngs)
                
                if num_left == 0 or num_right == 0 or num_depth == 0:
                    download_needed = True
                    break
                if num_left != num_right or num_left != num_depth:
                    download_needed = True
                    break
        
        if not download_needed:
            print(f"[skip] All raw files (images and depth maps) exist and match in counts for '{env_name}'. Skipping download step.")
        else:
            # Initialize tartanair directly to the final Google Drive root path
            print(f"[tartanair] Initializing download directory directly on Google Drive: {GDRIVE_TARTANGROUND_ROOT}")
            ta.init(GDRIVE_TARTANGROUND_ROOT)
            
            # Run multi-threaded download directly to Google Drive
            print(f"[tartanair] Launching download for '{env_name}' to fetch missing/raw files...")
            ta.download_ground_multi_thread(
                env=[env_name],
                version=VERSIONS,
                modality=MODALITIES,
                camera_name=CAMERAS,
                unzip=False,
                num_workers=8
            )
            
        # 2. Extract the downloaded zip files directly on Google Drive (recursive search for nested files)
        if not os.path.exists(dest_env_path):
            raise RuntimeError(f"Environment directory not found on Drive. Download might have failed for {env_name}.")
            
        zip_files = glob.glob(os.path.join(dest_env_path, "**/*.zip"), recursive=True)
        if len(zip_files) > 0:
            print(f"[extractor] Found {len(zip_files)} zip files on Drive under {dest_env_path}. Extracting in-place...")
            for zip_file in sorted(zip_files):
                parent_dir = os.path.dirname(zip_file)
                zip_name = os.path.basename(zip_file)
                folder_name = zip_name.replace('.zip', '')
                
                # Drop/delete right camera depth maps without unzipping
                if folder_name.startswith("depth_") and "rcam" in folder_name:
                    print(f"[extractor] Dropping/deleting right camera depth zip: {zip_name}")
                    try:
                        os.remove(zip_file)
                    except Exception as e:
                        print(f"[WARNING] Failed to remove {zip_file}: {e}")
                    continue
                    
                # Verify if the zip file contains the folder inside or if we need to create it
                with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                    namelist = zip_ref.namelist()
                    files = [x for x in namelist if not x.endswith('/')]
                    if len(files) == 0:
                        extract_dest = os.path.join(parent_dir, folder_name)
                    else:
                        first_file = files[0]
                        if '/' in first_file:
                            extract_dest = parent_dir
                        else:
                            extract_dest = os.path.join(parent_dir, folder_name)
                        
                print(f"[extractor] Extracting {zip_name} to {extract_dest}...")
                os.makedirs(extract_dest, exist_ok=True)
                cmd = f"unzip -q -o '{zip_file}' -d '{extract_dest}'"
                status = os.system(cmd)
                if status != 0:
                    print(f"[WARNING] Extraction command returned non-zero code for {zip_file}")
                else:
                    # Delete zip file after successful extraction to save space on Google Drive
                    os.remove(zip_file)
        else:
            print("[extractor] No zip files found. Skipping extraction step.")
                
        # 3. Preprocess depth files to .npy
        depth_pngs = glob.glob(os.path.join(dest_env_path, "**/depth_lcam_front/**/*.png"), recursive=True)
        if len(depth_pngs) > 0:
            preprocess_depth_files(dest_env_path)
        else:
            print("[preprocessor] No raw depth PNG files found for conversion. Skipping preprocessing step.")
        
        # 4. Visualize a sample from this environment as proof of correctness
        visualize_sample(dest_env_path)
        
        print(f"[success] Successfully processed '{env_name}'!")
            
    except Exception as e:
        print(f"[ERROR] Processing failed for environment '{env_name}': {e}")
        print("Stopping pipeline. Please check errors and resume.")
        break

# Final check: ensure drive is mounted
if not os.path.exists("/content/drive/MyDrive"):
    print("\n[gdrive] Remounting Google Drive...")
    drive.mount('/content/drive')
        
print("\n" + "=" * 80)
print("  PIPELINE PROCESSING COMPLETED")
print("=" * 80)
